# AutoIntel — BERT NER Fine-Tuning

Fine-tunes `yashpwr/resume-ner-bert-v2` on the `jjzha/skillspan` dataset for resume skill extraction.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

**Estimated time:** 30-45 minutes on T4 GPU

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers datasets seqeval accelerate
print('Done!')

In [ ]:
# CELL 2 — Load dataset
from datasets import load_dataset
import json

print('Loading jjzha/skillspan dataset...')
dataset = load_dataset('jjzha/skillspan')
print(dataset)

label_names = dataset['train'].features['labels'].feature.names
print('\nLabels:', label_names)

print('\nSample entry:')
sample = dataset['train'][0]
for tok, lbl in zip(sample['tokens'][:10], sample['labels'][:10]):
    print(f"  {tok:<20} {dataset['train'].features['labels'].feature.int2str(lbl)}")

In [ ]:
# CELL 3 — Load base model and tokenizer
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

BASE_MODEL = 'yashpwr/resume-ner-bert-v2'
OUTPUT_DIR = './fine_tuned_ner'

num_labels = len(label_names)
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f'Loading model...')
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

model = model.to(device)
print('Model ready!')

In [ ]:
# CELL 4 — Tokenize and align labels
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
        padding='max_length',
    )
    all_labels = []
    for i, word_labels in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                aligned.append(word_labels[word_id])
            else:
                aligned.append(-100)
            prev_word_id = word_id
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized

print('Tokenizing train split...')
train_dataset = dataset['train'].map(
    tokenize_and_align_labels, batched=True, batch_size=64,
    remove_columns=dataset['train'].column_names
)

print('Tokenizing test split...')
eval_dataset = dataset['test'].map(
    tokenize_and_align_labels, batched=True, batch_size=64,
    remove_columns=dataset['test'].column_names
)

train_dataset.set_format('torch')
eval_dataset.set_format('torch')

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('Done!')

In [ ]:
# CELL 5 — Evaluation metric
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        tl, tp = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                tl.append(id2label[l])
                tp.append(id2label[p])
        true_labels.append(tl)
        true_preds.append(tp)
    return {
        'f1':        f1_score(true_labels, true_preds),
        'precision': precision_score(true_labels, true_preds),
        'recall':    recall_score(true_labels, true_preds),
    }

print('Metric defined.')

In [ ]:
# CELL 6 — Train (this takes ~30-40 min on T4 GPU)
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# CELL 7 — Evaluate
results = trainer.evaluate()
print('\n' + '='*50)
print('RESULTS')
print('='*50)
print(f"F1:        {results['eval_f1']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")
print('='*50)

In [ ]:
# CELL 8 — Save model
import os, json

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metadata = {
    'base_model': BASE_MODEL,
    'dataset': 'jjzha/skillspan',
    'train_samples': len(train_dataset),
    'epochs': 4,
    'learning_rate': 2e-5,
    'eval_f1': round(results['eval_f1'], 4),
    'eval_precision': round(results['eval_precision'], 4),
    'eval_recall': round(results['eval_recall'], 4),
}
with open(os.path.join(OUTPUT_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved to {OUTPUT_DIR}:')
for fname in os.listdir(OUTPUT_DIR):
    mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024 / 1024
    print(f'  {fname}  ({mb:.1f} MB)')

In [ ]:
# CELL 9 — Quick test
from transformers import pipeline

ner_pipe = pipeline(
    'ner', model=OUTPUT_DIR, tokenizer=OUTPUT_DIR,
    aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1,
)

test_text = (
    'John dela Cruz is a Software Engineer with 3 years of experience. '
    'Skills: Python, React, Node.js, PostgreSQL, Docker, AWS. '
    'Bachelor of Science in Computer Science from De La Salle University.'
)

print('Test result:')
for ent in ner_pipe(test_text):
    print(f"  [{ent['entity_group']}]  '{ent['word']}'  ({ent['score']:.3f})")

In [ ]:
# CELL 10 — Download
import shutil
from google.colab import files

shutil.make_archive('fine_tuned_ner', 'zip', '.', 'fine_tuned_ner')
print('Downloading fine_tuned_ner.zip...')
files.download('fine_tuned_ner.zip')
print('Done! Unzip and place fine_tuned_ner/ inside backend/ in your project.')